In [1]:
import logging
import sys
from pathlib import Path
import pandas as pd


logging.basicConfig(
    format="%(asctime)s.%(msecs)d %(levelname)s %(filename)s:%(lineno)d %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger()
logger.setLevel(logging.INFO)


# Move up one level from scripts/ to find the project root
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

logging.info(f"Project root added to path: {PROJECT_ROOT}")

from src.core.dataset_parser import ClaudetteParser


14:58:21.256 INFO 1386925709.py:20 Project root added to path: /home/miguel/enhesa-tos-service


In [ ]:


zip_path = PROJECT_ROOT / "data" / "ToS.zip"
logging.info(f"Fetching raw data from {zip_path}& parsing...")

parser = ClaudetteParser(zip_path)


df = parser.parse()


# =====================================================================
# 5. EXECUTE THE SEMANTIC RETRIEVAL ENGINE & FREEZE STATE
# =====================================================================
from src.core.feature_engine import SimpleEmbeddingEngine

# Define the target directory where pre-computed states will be saved
ARTIFACT_DIR = PROJECT_ROOT / "models" / "search_artifacts"

logging.info("Initializing the Semantic Vector Space Engine...")
engine = SimpleEmbeddingEngine(model_name="all-MiniLM-L6-v2")

# 1. Compute the dense text-coordinate matrices
X_embeddings = engine.build_index(df)
logging.info(f"Successfully generated text embeddings matrix of shape: {X_embeddings.shape}")

# 2. Freeze the calculated numbers and mapping records to disk
logging.info(f"Saving compiled matrices and metadata definitions to {ARTIFACT_DIR}...")
engine.save_artifacts(ARTIFACT_DIR)

print(f"\n✅ Processing complete! Embedding binaries and indexes are safely frozen on disk.")







df.head()


14:58:21.266 INFO 1179697088.py:2 Fetching raw data from /home/miguel/enhesa-tos-service/data/ToS.zip& parsing...


📊 Dataset Loaded Successfully!
Total rows (clauses): 9414
Features: 12 columns

Target Class Breakdown ('is_unfair'):
is_unfair
0    1.0
Name: proportion, dtype: float64


,company,sentence_idx,text,is_unfair,arbitration,unilateral_change,content_removal,jurisdiction,choice_of_law,limitation_of_liability,unilateral_termination,contract_by_using
0,Fitbit,0,we recently revised these terms .,0,0,0,0,0,0,0,0,0
1,Fitbit,1,please review the summary of changes and updat...,0,0,0,0,0,0,0,0,0
2,Fitbit,2,you can find the earlier terms in our archive .,0,0,0,0,0,0,0,0,0
3,Fitbit,3,"updated : september 28 , 2017",0,0,0,0,0,0,0,0,0
4,Fitbit,4,"effective : october 30 , 2017 , unless you agr...",0,0,0,0,0,0,0,0,0


In [ ]:
# =====================================================================
# 6. VERIFY BY LOADING BACK IN LIKE AN API SERVER WOULD
# =====================================================================
print("\n🏁 Sanity Check: Simulating API server initialization from frozen assets...")

# Create a completely fresh runtime instance to prove we don't need to rebuild it
prod_engine = SimpleEmbeddingEngine(model_name="all-MiniLM-L6-v2")

# Warm up the engine memory blocks directly using the saved binary arrays
prod_engine.load_artifacts(ARTIFACT_DIR)

# Fire off a real-world test search query
test_query = "Can this app modify my contract terms whenever they feel like it?"
test_matches = prod_engine.search(query_text=test_query, top_k=2)

print(f"\n🔍 Query tested: '{test_query}'\n")
for idx, match in enumerate(test_matches, 1):
    print(f"Match #{idx} (Similarity alignment score: {match['similarity_score']:.4f})")
    print(f"  Company Origin: {match['company']}")
    print(f"  Extracted Text: '{match['text']}'")
    print(f"  Unfair Flag:    {match['is_unfair']}\n")


print(f"📊 Dataset Loaded Successfully!")
print(f"Total rows (clauses): {df.shape[0]}")
print(f"Features: {df.shape[1]} columns\n")
print("Target Class Breakdown ('is_unfair'):")
print(df['is_unfair'].value_counts(normalize=True))

/home/miguel/.cache/pypoetry/virtualenvs/enhesa-tos-service-eiCNO2e5-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
14:58:27.982 INFO 2412585105.py:9 Initializing the Semantic Vector Space Engine...
14:58:27.983 INFO feature_engine.py:20 Loading SentenceTransformer vector mapping space: all-MiniLM-L6-v2
14:58:27.985 INFO model.py:190 No device provided, using cpu
14:58:28.194 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
14:58:28.195 WARNING _http.py:928 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
14:58:28.215 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/api/resolve-cache


✅ Processing complete! Embedding binaries and indexes are safely frozen on disk.

🏁 Sanity Check: Simulating API server initialization from frozen assets...


14:59:57.143 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
14:59:57.166 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
14:59:57.306 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
14:59:57.325 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
14:59:57.327 INFO model.py:1001 Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
14:59:57.468 INFO _client.py:1025 HTTP Request: HEAD https://huggingface.


🔍 Query tested: 'Can this app modify my contract terms whenever they feel like it?'

Match #1 (Similarity alignment score: 0.6142)
  Company Origin: Endomondo
  Extracted Text: 'in such cases , modifications will be effective at the time of your agreement to the modified version of the terms .'
  Unfair Flag:    0

Match #2 (Similarity alignment score: 0.6056)
  Company Origin: PokemonGo
  Extracted Text: 'it 's important that you review the terms whenever we modify them , because if you continue to use the services after we have posted modified terms on the site or app , or otherwise communicate them to you , you are indicating to us that you agree to be bound by the modified terms .'
  Unfair Flag:    0



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from src.core.classifier import ClauseClassifier


# Extract the binary classification labels from your dataframe
y_targets = df['is_unfair'].values

# --- RUN EXPERIMENT A: LOGISTIC REGRESSION ---
lr_model = LogisticRegression(class_weight="balanced")
clf_lr = ClauseClassifier(estimator=lr_model)
metrics_lr = clf_lr.train_and_evaluate(X_embeddings, y_targets)

# --- RUN EXPERIMENT A: LOGISTIC REGRESSION ---
lr_model = LogisticRegression(class_weight="balanced")
clf_lr = ClauseClassifier(estimator=lr_model)
metrics_lr = clf_lr.train_and_evaluate(X_embeddings, y_targets)

# --- RUN EXPERIMENT B: RANDOM FOREST ---
rf_model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
clf_rf = ClauseClassifier(estimator=rf_model)
metrics_rf = clf_rf.train_and_evaluate(X_embeddings, y_targets)

# --- RUN EXPERIMENT C: SUPPORT VECTOR MACHINE (SVM) ---
svm_model = SVC(kernel="rbf", class_weight="balanced")
clf_svm = ClauseClassifier(estimator=svm_model)
metrics_svm = clf_svm.train_and_evaluate(X_embeddings, y_targets)

15:17:58.237 INFO classifier.py:32 Initialized wrapper with custom estimator: LogisticRegression


NameError: name 'y_targets' is not defined